# Exploratory Data Analysis — HelpDesk SLA Violation Prediction

This notebook explores the raw helpdesk ticket dataset from Mendeley to understand:
- Dataset size, column types, and missing values
- Distribution of the SLA violation target variable
- Distributions of key numeric features
- Correlations between features and the target
- Event history patterns per ticket

Dataset source: https://data.mendeley.com/datasets/btm76zndnt/2

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Reproducibility
np.random.seed(42)

# Paths — works whether you run from repo root or notebooks/
REPO_ROOT = (
    Path().resolve().parent
    if Path().resolve().name == "notebooks"
    else Path().resolve()
)
RAW_DIR = REPO_ROOT / "data" / "raw" / "HelpDeskTickets"

print(f"Data directory: {RAW_DIR}")
print(f"Exists: {RAW_DIR.exists()}")

## 1. Load Raw Data

In [ ]:
issues = pd.read_csv(RAW_DIR / "issues.csv")
history = pd.read_csv(RAW_DIR / "issues_change_history.csv")

print(f"issues shape:  {issues.shape}")
print(f"history shape: {history.shape}")

In [ ]:
issues.head(3)

In [ ]:
history.head(3)

## 2. Data Types and Missing Values

In [ ]:
print("=== issues.csv ===")
print(issues.dtypes)
print()
missing = issues.isnull().sum()
missing_pct = (missing / len(issues) * 100).round(2)
missing_report = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
(
    missing_report[missing_report["missing_count"] > 0].sort_values(
        "missing_pct", ascending=False
    )
)

In [ ]:
print("=== issues_change_history.csv ===")
print(history.dtypes)
print()
history.isnull().sum()

## 3. Target Variable — SLA Violation

We define SLA violation as: `wf_total_time > 7 * 24 * 3600` (7 days in seconds)

In [ ]:
SLA_THRESHOLD_SECONDS = 7 * 24 * 3600

issues["sla_violation"] = (issues["wf_total_time"] > SLA_THRESHOLD_SECONDS).astype(int)

violation_counts = issues["sla_violation"].value_counts()
violation_pct = issues["sla_violation"].value_counts(normalize=True) * 100

print("SLA Violation Distribution:")
print(pd.DataFrame({"count": violation_counts, "pct": violation_pct.round(2)}))

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(
    ["No Violation (0)", "Violation (1)"],
    violation_counts.values,
    color=["steelblue", "tomato"],
)
ax.set_title("SLA Violation Class Balance")
ax.set_ylabel("Ticket Count")
for i, v in enumerate(violation_counts.values):
    ax.text(
        i,
        v + 200,
        f"{v:,}\n({violation_pct.values[i]:.1f}%)",
        ha="center",
        fontsize=10,
    )
plt.tight_layout()
plt.show()

## 4. Numeric Feature Distributions

In [ ]:
numeric_cols = [
    "issue_contr_count",
    "issue_comments_count",
    "wf_total_time",
    "processing_steps",
]

print(issues[numeric_cols].describe().round(2))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    data = issues[col].dropna()
    # clip at 99th percentile for readability
    clip_val = data.quantile(0.99)
    axes[i].hist(
        data.clip(upper=clip_val),
        bins=50,
        color="steelblue",
        edgecolor="white",
        linewidth=0.3,
    )
    axes[i].set_title(f"{col} (clipped at 99th pct)")
    axes[i].set_xlabel("Value")
    axes[i].set_ylabel("Frequency")

plt.suptitle("Numeric Feature Distributions", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Categorical Feature Distributions

In [ ]:
for col in ["issue_type", "issue_priority", "issue_resolution", "issue_status"]:
    if col in issues.columns:
        print(f"\n--- {col} ---")
        print(issues[col].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col in zip(axes, ["issue_priority", "issue_type"], strict=True):
    counts = issues[col].value_counts()
    ax.bar(counts.index, counts.values, color="steelblue")
    ax.set_title(f"{col} distribution")
    ax.set_xlabel(col)
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## 6. SLA Violation Rate by Categorical Feature

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, col in zip(axes, ["issue_priority", "issue_type"], strict=True):
    rate = issues.groupby(col)["sla_violation"].mean().sort_values(ascending=False)
    ax.bar(rate.index, rate.values * 100, color="tomato")
    ax.set_title(f"SLA Violation Rate by {col}")
    ax.set_ylabel("Violation Rate (%)")
    ax.set_xlabel(col)
    ax.tick_params(axis="x", rotation=30)
    for j, v in enumerate(rate.values):
        ax.text(j, v * 100 + 0.5, f"{v * 100:.1f}%", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## 7. Feature Correlations with Target

In [ ]:
# Compute event counts from history
history["created"] = pd.to_datetime(history["created"], errors="coerce")
event_counts = history.groupby("issueid").size().reset_index(name="num_events")

ticket_durations = (
    history.groupby("issueid")["created"].agg(["min", "max"]).reset_index()
)
ticket_durations["duration_seconds"] = (
    ticket_durations["max"] - ticket_durations["min"]
).dt.total_seconds()

issues = issues.merge(event_counts, left_on="id", right_on="issueid", how="left")
issues = issues.merge(
    ticket_durations[["issueid", "duration_seconds"]],
    left_on="id",
    right_on="issueid",
    how="left",
)
issues["num_events"] = issues["num_events"].fillna(0)
issues["duration_seconds"] = issues["duration_seconds"].fillna(0)

corr_cols = [
    "issue_contr_count",
    "issue_comments_count",
    "wf_total_time",
    "processing_steps",
    "num_events",
    "duration_seconds",
    "sla_violation",
]
corr_matrix = issues[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr_matrix.values, cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(corr_cols, fontsize=9)
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(
            j,
            i,
            f"{corr_matrix.values[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=8,
        )
ax.set_title("Feature Correlation Matrix", fontsize=12)
plt.tight_layout()
plt.show()

## 8. Event History — Events per Ticket

In [ ]:
print("Event counts per ticket (summary):")
print(event_counts["num_events"].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribution of event counts
axes[0].hist(
    event_counts["num_events"].clip(upper=event_counts["num_events"].quantile(0.99)),
    bins=40,
    color="steelblue",
    edgecolor="white",
    linewidth=0.3,
)
axes[0].set_title("Events per Ticket (clipped at 99th pct)")
axes[0].set_xlabel("Number of Events")
axes[0].set_ylabel("Frequency")

# Event counts by violation status
merged = issues[["id", "sla_violation", "num_events"]].dropna()
for label, grp in merged.groupby("sla_violation"):
    axes[1].hist(
        grp["num_events"].clip(upper=grp["num_events"].quantile(0.99)),
        bins=30,
        alpha=0.6,
        label=f"{'Violation' if label else 'No Violation'}",
        density=True,
    )
axes[1].set_title("Event Count Distribution by SLA Outcome")
axes[1].set_xlabel("Number of Events")
axes[1].set_ylabel("Density")
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Unique Change Fields in History

In [ ]:
print("Unique fields tracked in change history:")
print(history["field"].value_counts())

## 10. Key Observations

1. **Class balance**: The dataset may be imbalanced. Check the violation rate above — if it is below ~30%, we should monitor precision/recall closely and consider class-weighted training.
2. **wf_total_time is right-skewed**: Most tickets resolve quickly; a long tail drives violations. Log-transforming this feature (done in `build_features.py`) helps linear models.
3. **issue_priority is a strong signal**: Higher-priority tickets tend to get faster resolution, but certain categories show elevated violation rates.
4. **num_events varies widely**: Tickets with very high event counts often indicate back-and-forth escalations and correlate with SLA breaches.
5. **Missing values**: Several `wf_*` columns contain zeros/NaNs for workflows that were never entered. This is expected and handled by the preprocessing pipeline.

These observations motivate the engineered features in `build_features.py`: `events_per_day`, `log_total_time`, `comments_per_contributor`, and `is_high_priority`.